## **Recomendação Content-Based (Item ↔ Item)**

**Diferença Fundamental**

| Abordagem | O que compara | Pergunta-chave |
|---|---|---|
| **Content-Based (Perfil ↔ Item)** | Perfil do usuário × Features dos itens | "Este item combina com o que o usuário gosta?" |
| **Content-Based (Item ↔ Item)**  | Features do item-alvo × Features dos outros itens | "Quem gostou deste item, vai gostar destes similares?" |

## **Catálogo de filmes**

In [5]:
import pandas as pd
import numpy as np
filmes = pd.DataFrame({
    'item_id':      list(range(1, 21)),
    'titulo': [
        'Matrix', 'Interestelar', 'Vingadores', 'John Wick', 'Duro de Matar',
        'Mad Max', 'Toy Story', 'Shrek', 'Procurando Nemo', 'Divertida Mente',
        'Titanic', 'La La Land', 'Diário de uma Paixão', 'O Poderoso Chefão',
        'Forrest Gump', 'Clube da Luta', 'Se Beber Não Case',
        'Todo Mundo em Pânico', 'Deadpool', 'Interestelar 2'
    ],
    'genero': [
        'Ficção', 'Ficção', 'Ficção', 'Ação', 'Ação',
        'Ação', 'Animação', 'Animação', 'Animação', 'Animação',
        'Romance', 'Romance', 'Romance', 'Drama', 'Drama',
        'Drama', 'Comédia', 'Comédia', 'Comédia', 'Ficção'
    ],
    'subgenero': [
        'Ação', 'Drama', 'Ação', 'Suspense', 'Suspense',
        'Aventura', 'Comédia', 'Comédia', 'Aventura', 'Drama',
        'Drama', 'Musical', 'Drama', 'Crime', 'Romance',
        'Suspense', 'Aventura', 'Paródia', 'Ação', 'Drama'
    ],
    'ano': [1999, 2014, 2012, 2014, 1988, 2015, 1995, 2001, 2003, 2015,
            1997, 2016, 2004, 1972, 1994, 1999, 2009, 2000, 2016, 2025],
    'duracao': [136, 169, 143, 101, 132, 120, 81, 90, 100, 95,
                194, 128, 123, 175, 142, 139, 100, 88, 108, 170],
    'classificacao': [14, 12, 12, 16, 16, 16, 0, 0, 0, 0,
                      12, 12, 12, 16, 12, 18, 16, 16, 16, 12],
    'nota_media': [4.8, 4.9, 4.5, 4.6, 4.4, 4.7, 4.9, 4.8, 4.7, 4.9,
                   4.5, 4.6, 4.7, 5.0, 4.8, 4.8, 4.3, 4.0, 4.5, 4.7]
})


In [6]:

print(f"Catálogo: {filmes.shape[0]} filmes × {filmes.shape[1]} colunas")
print(filmes.head())

Catálogo: 20 filmes × 8 colunas
   item_id         titulo  genero subgenero   ano  duracao  classificacao  \
0        1         Matrix  Ficção      Ação  1999      136             14   
1        2   Interestelar  Ficção     Drama  2014      169             12   
2        3     Vingadores  Ficção      Ação  2012      143             12   
3        4      John Wick    Ação  Suspense  2014      101             16   
4        5  Duro de Matar    Ação  Suspense  1988      132             16   

   nota_media  
0         4.8  
1         4.9  
2         4.5  
3         4.6  
4         4.4  


## **Pré-Prcessamento**

In [7]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### **Features categóricas**

In [8]:
cat_cols = ['genero', 'subgenero']

ohe = OneHotEncoder(sparse_output=False)
X_cat = ohe.fit_transform(filmes[cat_cols])
cat_names = ohe.get_feature_names_out(cat_cols)

### **Features numéricas**

In [9]:
scaler = StandardScaler()
X_num = scaler.fit_transform(filmes[['ano', 'duracao', 'classificacao']])

###  **Concatenar**

In [12]:
X_itens = np.hstack([X_num, X_cat])


In [11]:
feature_names = ['ano_scaled', 'duracao_scaled', 'classificacao_scaled'] + list(cat_names)

## **Montando o Dataframe**

In [13]:
X_itens_df = pd.DataFrame(X_itens, columns=feature_names, index=filmes['item_id'])

In [14]:
print(f"Matriz de features: {X_itens_df.shape}")

Matriz de features: (20, 18)


In [15]:
print(X_itens_df.round(3).head())

         ano_scaled  duracao_scaled  classificacao_scaled  genero_Animação  \
item_id                                                                      
1            -0.453           0.296                 0.433              0.0   
2             0.805           1.347                 0.100              0.0   
3             0.637           0.519                 0.100              0.0   
4             0.805          -0.818                 0.766              0.0   
5            -1.376           0.169                 0.766              0.0   

         genero_Ação  genero_Comédia  genero_Drama  genero_Ficção  \
item_id                                                             
1                0.0             0.0           0.0            1.0   
2                0.0             0.0           0.0            1.0   
3                0.0             0.0           0.0            1.0   
4                1.0             0.0           0.0            0.0   
5                1.0             0.0   

## **Matriz de Similaridade Item x Item**

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
sim_matrix = cosine_similarity(X_itens_df)



In [18]:
sim_matrix

array([[ 1.        ,  0.32348232,  0.73933733, -0.08841856,  0.30051558,
        -0.0459945 , -0.19698578, -0.24848177, -0.25741084, -0.35535702,
         0.23035431, -0.14218073,  0.01052753,  0.36482024,  0.21329982,
         0.2680264 , -0.03231804,  0.04134419,  0.23017774,  0.16174253],
       [ 0.32348232,  1.        ,  0.64121856, -0.09026656, -0.17900561,
         0.12910281, -0.45579728, -0.35492063, -0.26838343,  0.02892964,
         0.60757462,  0.23345665,  0.2739012 , -0.00541645, -0.0098811 ,
         0.06826735, -0.19267838, -0.43170006,  0.01391906,  0.94680308],
       [ 0.73933733,  0.64121856,  1.        ,  0.05099628, -0.20483268,
         0.17566119, -0.3057835 , -0.22473257, -0.17103187, -0.03291876,
         0.16771433,  0.23120362, -0.03110771, -0.14897854, -0.10316224,
         0.00788995, -0.03895035, -0.23649932,  0.42944324,  0.65688169],
       [-0.08841856, -0.09026656,  0.05099628,  1.        ,  0.31970578,
         0.67764742, -0.1575015 , -0.13879291, -

In [19]:
sim_df = pd.DataFrame(sim_matrix, index=filmes['titulo'], columns=filmes['titulo'])

In [20]:
sim_df

titulo,Matrix,Interestelar,Vingadores,John Wick,Duro de Matar,Mad Max,Toy Story,Shrek,Procurando Nemo,Divertida Mente,Titanic,La La Land,Diário de uma Paixão,O Poderoso Chefão,Forrest Gump,Clube da Luta,Se Beber Não Case,Todo Mundo em Pânico,Deadpool,Interestelar 2
titulo,,,,,,,,,,,,,,,,,,,,
Matrix,1.000000,0.323482,0.739337,-0.088419,0.300516,-0.045995,-0.196986,-0.248482,-0.257411,-0.355357,0.230354,-0.142181,0.010528,0.364820,0.213300,0.268026,-0.032318,0.041344,0.230178,0.161743
Interestelar,0.323482,1.000000,0.641219,-0.090267,-0.179006,0.129103,-0.455797,-0.354921,-0.268383,0.028930,0.607575,0.233457,0.273901,-0.005416,-0.009881,0.068267,-0.192678,-0.431700,0.013919,0.946803
Vingadores,0.739337,0.641219,1.000000,0.050996,-0.204833,0.175661,-0.305784,-0.224733,-0.171032,-0.032919,0.167714,0.231204,-0.031108,-0.148979,-0.103162,0.007890,-0.038950,-0.236499,0.429443,0.656882
John Wick,-0.088419,-0.090267,0.050996,1.000000,0.319706,0.677647,-0.157502,-0.138793,-0.171574,0.016142,-0.416722,0.243063,0.051910,-0.411983,-0.298904,0.310043,0.433587,0.319004,0.476767,0.065445
Duro de Matar,0.300516,-0.179006,-0.204833,0.319706,1.000000,0.083525,-0.100443,-0.223456,-0.268744,-0.492668,0.230214,-0.343618,0.034042,0.614799,0.368973,0.631303,-0.022057,0.202808,-0.203447,-0.370819
Mad Max,-0.045995,0.129103,0.175661,0.677647,0.083525,1.000000,-0.345312,-0.296995,-0.081141,-0.089163,-0.190656,0.293128,0.027304,-0.331938,-0.250210,0.101917,0.613658,0.136959,0.432861,0.271403
Toy Story,-0.196986,-0.455797,-0.305784,-0.157502,-0.100443,-0.345312,1.000000,0.981681,0.815875,0.683124,-0.369128,-0.204712,0.001971,-0.152787,-0.042099,-0.421637,-0.097186,0.105732,-0.237999,-0.469057
Shrek,-0.248482,-0.354921,-0.224733,-0.138793,-0.223456,-0.296995,0.981681,1.000000,0.842211,0.764947,-0.358539,-0.112884,-0.011258,-0.265563,-0.110791,-0.481707,-0.115728,0.016491,-0.197966,-0.328884
Procurando Nemo,-0.257411,-0.268383,-0.171032,-0.171574,-0.268744,-0.081141,0.815875,0.842211,1.000000,0.781529,-0.291228,-0.078314,-0.023910,-0.276217,-0.114818,-0.497722,0.047489,-0.070192,-0.214068,-0.236607


## **Aplicação — KNN Item ↔ Item**

In [21]:
from sklearn.neighbors import NearestNeighbors

def recomendar_itens_similares(item_id, X_itens_df, filmes, k=5):
    """Dado um item, retorna os K itens mais similares."""
    # KNN
    knn = NearestNeighbors(n_neighbors=k+1, metric='cosine', algorithm='brute')
    knn.fit(X_itens_df)

    # Buscar vizinhos do item-alvo
    idx = filmes[filmes['item_id'] == item_id].index[0]
    dist, indices = knn.kneighbors(X_itens_df.iloc[[idx]])
    sims = 1 - dist[0]

    # Montar resultado
    recs = filmes.iloc[indices[0]].copy()
    recs['similaridade'] = sims
    recs = recs[recs['item_id'] != item_id]  # remover o próprio item
    return recs.sort_values('similaridade', ascending=False).head(k)



In [32]:

# Exemplo: itens similares a "Matrix"
print("Filmes similares a 'Matrix':")
recs = recomendar_itens_similares(1, X_itens_df, filmes, k=5)
print(recs[['titulo', 'genero', 'subgenero', 'similaridade']].to_string(index=False))

Filmes similares a 'Matrix':
           titulo genero subgenero  similaridade
       Vingadores Ficção      Ação      0.739337
O Poderoso Chefão  Drama     Crime      0.364820
     Interestelar Ficção     Drama      0.323482
    Duro de Matar   Ação  Suspense      0.300516
    Clube da Luta  Drama  Suspense      0.268026
